<h1> Step 7: MCTS (Monte Carlo Tree Search) </h1>

MCTS has four stages

*1) Selection* </br>
We move from Root to the best Child: </br>
With the criterion: UCT

*2) Expansion* </br>
If we have a new Node:</br>
We add a Child.

*3) Simulation (Rollout)* </br>
From that State:</br>
We simulate until the end of the game.</br>
For example:</br>
Random Play:</br>
X → O → X → ...

*4) Backpropagation* </br>
We return the result:</br>
Win</br>
Loss</br>
Draw</br>

And update the Statistics.

### Upper Confidence Bound for Trees (UCT)

$UCT_i=  Q_i/N_i   +C√(ln⁡〖N_p 〗/N_i )$

where:

- $Q_i$: total reward (wins) obtained from child node \(i\)
- $N_i$: number of visits to child node \(i\)
- $N_p$: number of visits to the parent node
- $C$: exploration constant (usually $C=\sqrt{2}$)

| Symbol        | Meaning              |
| ------------- | -------------------- |
| Q             | Wins                 |
| N             | Visits               |
| C             | Exploration constant |
| Parent Visits | Number of parent visits |


If: child.visits == 0</br>
then: we set the UCT value to:</br>
$ +\infty $ </br>
because we want each move to be tried at least once.

</br></br>
MCTS =
Exploitation + Exploration

And the same idea is later seen again in:</br>
Reinforcement Learning</br>
Bandit Algorithms.

This is exactly the difference between the MCTS philosophy and Greedy Search.

Greedy says: </br>
Choose the best thing I see now.

But MCTS says:</br>
Both continue the best current path and do not miss the opportunity to discover unknown paths.

Here we have a paradigm shift. So far: 

*Minimax / Alpha-Beta* (Exact) </br>
→ Model-based</br>
Game tree exploration</br>
→ Exact value calculation


*But MCTS:* (Estimate - Probability) </br>
Monte Carlo Tree Search</br>
→ Simulation-based</br>
→ Future sampling</br>
→ Best move estimation

MCTS is an alternative to Minimax in environments where: 

The state space is too large.</br>
A complete search is not possible.</br>
A statistical estimate is sufficient.


Minimax says: “Count all the important futures.”</br>
MCTS says: “Estimate the future with thousands of random simulations.”

**MCTS itself is not a learning algorithm; it is a planning/search algorithm.**
**But it can be used alongside RL and Neural Networks to build a learning system.**

MCTS does not learn anything from previous experiences.

What it does:

Takes a current state</br>
Builds the game tree</br>
Performs simulations</br>
Stores node statistics</br>
Chooses the best current move</br>

Using UCT, MCTS strikes a balance between:

**Exploitation** </br>
and:</br>
**Exploration**


So why do we say MCTS is intelligent?

Because it does intelligent search, not learning.</br>

Difference:</br>

Search:</br>
I have a model of the environment; I find the best decision.

Learning:</br>
I don't have a model of the environment; I learn from experience.

We shouldn't confuse MCTS with RL or Neural Network!!!

| | MCTS | RL | Neural Network |
| ------------------- | ----------------- | ------------------ | --------------------- |
| Type | Search | Learning Framework | Function Approximator |
| Learns? | ❌ Usually not | ✅ Yes | ✅ With training |
| Requires experience? | Not necessarily | Yes | Yes |
| Has long-term memory? | No | Yes | Yes |
| Goal | Current best decision | Policy learning | Function approximation |
| Example | Go Search | Q-Learning | CNN, Transformer |

What is the role of MCTSNode?

In Minimax we only had a temporary tree during Search. </br>
But in MCTS we store the tree.

In [21]:
from src.games import TicTacToe

from src.metrics import (
    SearchStats
)
from src.mcts import (
    MCTSNode,
    select,
    expand,
    simulate,
)

In [22]:
game = TicTacToe()

state = game.initial_state()

root = MCTSNode(
    game,
    state,
)

print(root.state)
print(root.visits)
print(root.wins)
print(root.untried_actions)

((' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' '), 'X')
0
0
[0, 1, 2, 3, 4, 5, 6, 7, 8]


In MCTS, unlike Minimax:

We don't build the whole tree first.

Initially:

Root

children = []

Then it grows slowly with Expansion.

<h3> Step 7.1: Selection (with UCT) </h3>

In [23]:
child1 = MCTSNode(
    game,
    game.result(
        state,
        0,
    ),
    parent=root,
    action=0,
)

child2 = MCTSNode(
    game,
    game.result(
        state,
        4,
    ),
    parent=root,
    action=4,
)

root.children = [
    child1,
    child2,
]

root.untried_actions.remove(0)
root.untried_actions.remove(4)

In [24]:
child1.visits = 10
child1.wins = 5

child2.visits = 10
child2.wins = 9

root.visits = 20

selected = select(root)

selected.action

In [25]:
print(selected.action)

print(
    root.fully_expanded()
)

None
False


In [26]:
root.untried_actions = []

print(
    root.fully_expanded()
)

True


In [27]:
selected = select(root)

print(selected.action)

4


<h3> Step 7.2: Expansion </h3>

The first stage where the MCTS tree actually grows.

We select a new move from the untested moves and create a new Child Node.

MCTS is a gradual search. In each iteration we choose a path.

Logic:

If there is an untried move:

Take one. </br>
Create a new state.</br>
Create a child.</br>
Add to children.</br>
Return child.

In [28]:
game = TicTacToe()

state = game.initial_state()

root = MCTSNode(
    game,
    state,
)

print(len(root.children))
print(root.untried_actions)

0
[0, 1, 2, 3, 4, 5, 6, 7, 8]


In [29]:
child = expand(root)

print(len(root.children))

print(root.children[0].action)

print(root.untried_actions)

1
8
[0, 1, 2, 3, 4, 5, 6, 7]


<h3> Step 7.3: Simulation / Rollout </h3>

We simulate a game from this state to the end. By choosing randomly. </br>
Why Random?</br>
In the basic version of MCTS:</br>
We don't have a smart policy.</br>

So:</br>
Simulation = Random Rollout</br>

We can improve it later:</br>
Heuristic rollout</br>
Learned policy</br>
Neural network</br>
But for now we will build the basic version.

Logic:

Copy the current state.</br>
Until the game is over:</br>
Choose a random action.</br>
Advance the state.</br>
Return the utility.

In [30]:
game = TicTacToe()

state = game.initial_state()

root = MCTSNode(
    game,
    state,
)

child = expand(root)

result = simulate(child)

result

1